In [0]:
# Extract Data From Source
# Called once per table by the For Each task
# Reads from Azure SQL via Connection Manager
# Writes to staging Delta table

dbutils.widgets.text("catalog_name", "")
dbutils.widgets.text("source_catalog", "")
dbutils.widgets.text("source_schema", "")
dbutils.widgets.text("staging_schema", "")
dbutils.widgets.text("table_name", "")

catalog_name   = dbutils.widgets.get("catalog_name")
source_catalog = dbutils.widgets.get("source_catalog")
source_schema  = dbutils.widgets.get("source_schema")
staging_schema = dbutils.widgets.get("staging_schema")
table_name     = dbutils.widgets.get("table_name")

print(f"Processing table: {table_name}")

In [0]:
# Step 1: Read from Azure SQL

source_path = f"{source_catalog}.{source_schema}.{table_name}"
df = spark.read.table(source_path)
source_count = df.count()
print(f"Read {source_count} rows from {source_path}")

In [0]:
# STEP 2: Write to staging

table_lower = table_name.lower()
staging_path = f"{catalog_name}.{staging_schema}.{table_lower}"

#debugging
print(f"Writing to: {staging_path}")

df.write.format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable(staging_path)

print(f"Written {source_count} rows to {staging_path}")

In [0]:
# STEP 3: Validate row counts match

staging_count = spark.read.table(staging_path).count()

if source_count == staging_count:
    print(f"Row count validated: {source_count} rows")
else:
    raise Exception(f"Row count mismatch! Source: {source_count} | Staging: {staging_count}")

In [0]:
#verification of staging table

print(f"Staging table preview — {table_name}:")
display(spark.sql(f"SELECT * FROM {catalog_name}.{staging_schema}.{table_lower} LIMIT 5"))
print(f"Row count: {source_count}")